# Day 5 — Delta: History, Time Travel, Optimize & Ops

Allow about 1.5–2 hours. Sections on CDF (6b), CHECK (7b), and the recap quiz can be done later if you run short on time.

Use notebook 01 first so `P_BASIC` and `P_PART` exist.

You will inspect history, use version and timestamp time travel, call `DeltaTable.history`, read about RESTORE and shallow clone (commented patterns), run OPTIMIZE and optional ZORDER, review VACUUM and CDF concepts, try optional hands-on cells for CDF and CHECK where your runtime allows, and work through extra drills and a short quiz.

Standard Delta versioning and optimization labs from Databricks training map well to this content.

## Setup

In [ ]:
%run ./02-Mount-Azure-Data-Lake

In [ ]:
BASE_PATH = base_path
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
P_PART = DAY5 + "/delta/flight_partitioned"
print("Using:", P_BASIC)

## Section 1 — Deep **history** inspection

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").show(50, truncate=False)

In [ ]:
# Latest version number
from pyspark.sql.functions import col as C

h = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`")
ver_max = h.selectExpr("max(version) as v").collect()[0]["v"]
ver_min = h.selectExpr("min(version) as v").collect()[0]["v"]
print("version min/max:", ver_min, ver_max)

## Section 2 — **Time travel** reads

In [ ]:
from pyspark.sql.functions import col

v = int(spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").selectExpr("max(version) as v").collect()[0]["v"])
if v >= 1:
    print("sum(count) at version v-1:")
    spark.read.format("delta").option("versionAsOf", v - 1).load(P_BASIC).groupBy().sum("count").show()
print("sum(count) at latest version", v)
spark.read.format("delta").option("versionAsOf", v).load(P_BASIC).groupBy().sum("count").show()

In [ ]:
from pyspark.sql.functions import col as C

row = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").filter(C("version") == 0).select("timestamp").collect()
if row:
    ts = str(row[0]["timestamp"])
    c_ts = spark.read.format("delta").option("timestampAsOf", ts).load(P_BASIC).count()
    print("Row count TIMESTAMP AS OF version-0 time:", c_ts)
else:
    print("No history row for version 0")

## Section 2b — **Subtract** rows: latest vs previous version

In [ ]:
from pyspark.sql.functions import col

h = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`")
v = int(h.selectExpr("max(version) as v").collect()[0]["v"])
cols = ["DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count"]
cur = spark.read.format("delta").option("versionAsOf", v).load(P_BASIC).select(*cols)
if v >= 1:
    prev = spark.read.format("delta").option("versionAsOf", v - 1).load(P_BASIC).select(*cols)
    print("counts prev, cur:", prev.count(), cur.count())
    print("in cur but not prev:", cur.subtract(prev).count())
    print("in prev but not cur:", prev.subtract(cur).count())
else:
    print("Only one version; run notebook 01 append cells first.")

## Section 2c — **`DeltaTable` API** for history (same log as SQL)

In [ ]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, P_BASIC).history(15).select("version", "timestamp", "operation").show(15, truncate=False)

## Section 3 — **RESTORE** (Databricks SQL)

Restore table state to an older version **creates a new version** that matches old data.

```sql
-- RESTORE TABLE delta.`abfss://.../path` TO VERSION AS OF 0;
```

Do not run RESTORE on shared class tables unless your workspace policy allows it; use a personal scratch path if you experiment.

## Section 3b — **SHALLOW CLONE** (pattern; optional)

Cheap **metadata** copy pointing at same files — great for experimentation / sharing snapshots.

```sql
-- CREATE TABLE delta.`abfss://.../day5/delta/clone_basic` SHALLOW CLONE delta.`abfss://.../day5/delta/flight_summary_basic`;
```

Requires write permission on the clone path. In shared training workspaces, use a path under your own `day5` prefix rather than a common table.

## Section 4 — **OPTIMIZE** + **ZORDER** (Databricks)

In [ ]:
# File metrics before OPTIMIZE
spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`").select("numFiles", "sizeInBytes").show(truncate=False)

In [ ]:
# Compacts small files (Databricks Delta)
spark.sql(f"OPTIMIZE delta.`{P_PART}`")

In [ ]:
# Z-order by high-cardinality filter columns (example)
try:
    spark.sql(f"OPTIMIZE delta.`{P_PART}` ZORDER BY (DEST_COUNTRY_NAME)")
except Exception as e:
    print("ZORDER may require Photon/Delta version:", type(e).__name__, e)

In [ ]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`").select("numFiles", "sizeInBytes").show(truncate=False)

## Section 4b — **OPTIMIZE** on **`P_BASIC`** (compare detail)

In [ ]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").select("numFiles", "sizeInBytes").show(truncate=False)
spark.sql(f"OPTIMIZE delta.`{P_BASIC}`")
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").select("numFiles", "sizeInBytes").show(truncate=False)

## Section 4c — **Auto Optimize** (workspace / table settings — theory)

Many workspaces enable **automatic** `OPTIMIZE` / compaction policies on **managed** tables. For **external** `abfss://` paths (this course), you often **schedule** `OPTIMIZE` in **Jobs** after loads.

Discuss: **file size targets**, **cost vs query latency**, **concurrent writers**.

## Section 5 — **VACUUM** (discussion only)

`VACUUM` deletes data files **no longer referenced** by the table. Default **7-day** retention for time travel.

```python
# spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
# spark.sql(f"VACUUM delta.`{P_BASIC}` RETAIN 168 HOURS")
```

**Do not** run aggressive vacuum in class without admin sign-off.

## Section 6 — **Change Data Feed (CDF)** overview

CDF records row-level changes (`insert`, `update_preimage`, `update_postimage`, `delete`) for incremental downstream consumers.

Enable at table properties (example):

```sql
-- ALTER TABLE delta.`path` SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
```

Then `table_changes('path')` or Spark `readStream.format('delta').option('readChangeFeed','true')` — detailed in Day 6 / streaming modules.

## Section 6b — **Change Data Feed** — hands-on (optional; try/except)

In [ ]:
from pyspark.sql.functions import lit

P_CDF = DAY5 + "/delta/cdf_lab_demo"
base = spark.read.format("delta").load(P_BASIC).select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count").limit(30)
base.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_CDF)
try:
    spark.sql(f"ALTER TABLE delta.`{P_CDF}` SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print("CDF enabled on", P_CDF)
except Exception as e:
    print("ALTER TBLPROPERTIES:", type(e).__name__, str(e)[:200])

In [ ]:
# Append rows to generate INSERT change records (after CDF enabled)
from pyspark.sql.functions import col, lit

try:
    extra = spark.read.format("delta").load(P_CDF).limit(3).withColumn("count", col("count") + lit(1))
    extra.write.format("delta").mode("append").save(P_CDF)
    print("Append OK — new history version should exist")
    spark.sql(f"DESCRIBE HISTORY delta.`{P_CDF}`").select("version", "operation").show(5, truncate=False)
except Exception as e:
    print("CDF append demo:", type(e).__name__, e)

In [ ]:
# Read change feed from version 0 (API varies by DBR)
try:
    ch = spark.read.format("delta").option("readChangeFeed", "true").option("startingVersion", 0).load(P_CDF)
    print("Change feed schema:")
    ch.printSchema()
    ch.show(15, truncate=False)
except Exception as e:
    print("readChangeFeed:", type(e).__name__, str(e)[:220])
    print("→ Fallback: use DESCRIBE HISTORY + streaming module on Day 6")

In [ ]:
# SQL: table_changes (Databricks SQL — if available)
try:
    spark.sql(f"SELECT * FROM table_changes(delta.`{P_CDF}`, 0) LIMIT 20").show(truncate=False)
except Exception as e:
    print("table_changes SQL:", type(e).__name__, str(e)[:200])

## Section 7 — **CHECK** / **NOT NULL** constraints (Delta)

Delta supports **table constraints** on newer runtimes:

```sql
-- ALTER TABLE delta.`path` ADD CONSTRAINT cnt_nonneg CHECK (count >= 0);
```

Violations fail writes — great for Silver/Gold quality gates.

## Section 7b — Add a **CHECK** constraint (scratch table; try/except)

In [ ]:
P_CHK = DAY5 + "/delta/check_constraint_demo"
spark.read.format("delta").load(P_BASIC).limit(100).write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(P_CHK)
try:
    spark.sql(f"ALTER TABLE delta.`{P_CHK}` ADD CONSTRAINT cnt_nonneg CHECK (count >= 0)")
    print("Constraint added (or already exists)")
except Exception as e:
    print("ADD CONSTRAINT:", type(e).__name__, str(e)[:200])

In [ ]:
# Violation should fail the write
from pyspark.sql import Row

try:
    bad = spark.createDataFrame([Row(DEST_COUNTRY_NAME="X", ORIGIN_COUNTRY_NAME="Y", count=-1)])
    bad.write.format("delta").mode("append").save(P_CHK)
    print("Unexpected: negative row accepted")
except Exception as e:
    print("Expected failure on CHECK:", type(e).__name__, str(e)[:160])

## Section 8 — Repeated **history** after a small write

In [ ]:
from pyspark.sql.functions import lit

spark.read.format("delta").load(P_BASIC).limit(1).withColumn("audit_note", lit("history_demo")).write.format("delta").mode("append").option("mergeSchema", "true").save(P_BASIC)
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").select("version", "operation").show(5, truncate=False)

## Section 9 — Practice prompts

### P9.1 — Compare **row count** at **version 0** vs **latest** on `P_BASIC`. When do they differ?

### P9.2 — Run **OPTIMIZE** on `P_BASIC` and re-check **`DESCRIBE DETAIL`** (file counts may change).

### P9.3 — Explain why **time travel** protects you from bad **overwrite** during testing.

### P9.4 — Use **`DeltaTable.forPath`** + **`.history()`** to print **`operationMetrics`** for the last **MERGE** or **WRITE** you performed (from notebook **04** if available).

### P9.5 — Sketch a **Job** that runs **`OPTIMIZE`** then **`ANALYZE TABLE`** (if SQL warehouse) for a curated Gold table.

### P9.6 — What is the **default** retention for **`VACUUM`** and why is **7 days** the common safety margin?

## Section 10 — **Worked examples** (history + detail drills)

In [ ]:
# Drill A — last 3 operations with userMetadata if set
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").select("version", "operation", "operationMetrics").show(3, truncate=False)

In [ ]:
# Drill B — partition columns + minReaderVersion / minWriterVersion (if columns exist in DETAIL output)
d = spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`")
d.show(truncate=False)

In [ ]:
# Drill C — read same path with SQL vs DataFrame reader (row count parity)
sql_c = spark.sql(f"SELECT COUNT(*) c FROM delta.`{P_BASIC}`").collect()[0]["c"]
df_c = spark.read.format("delta").load(P_BASIC).count()
print(sql_c, df_c)

## Section 10d — More **history** / **detail** drills (run all)

In [ ]:
# Drill D — operations that touched P_PART in last 8 commits
spark.sql(f"DESCRIBE HISTORY delta.`{P_PART}`").select("version", "timestamp", "operation").show(8, truncate=False)

In [ ]:
# Drill E — min/max Spark reader versions (if present in DETAIL)
d = spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`")
cols = [c for c in d.columns if "version" in c.lower() or "format" in c.lower()]
if cols:
    d.select(*cols).show(truncate=False)
else:
    d.show(3, truncate=False)

In [ ]:
# Drill F — time travel: full schema at version 0 vs latest
h = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`")
v0 = h.selectExpr("min(version) as v").collect()[0]["v"]
v1 = h.selectExpr("max(version) as v").collect()[0]["v"]
print("versions", v0, v1)
if v0 is not None:
    spark.read.format("delta").option("versionAsOf", int(v0)).load(P_BASIC).printSchema()
print("--- latest ---")
spark.read.format("delta").load(P_BASIC).printSchema()

In [ ]:
# Drill G — ZORDER then re-check partition stats (P_PART)
try:
    spark.sql(f"OPTIMIZE delta.`{P_PART}` ZORDER BY (DEST_COUNTRY_NAME)")
except Exception as e:
    print("ZORDER:", type(e).__name__)
spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`").select("numFiles", "sizeInBytes").show(truncate=False)

## Section 11 — Recap quiz

Short answers (discuss in class or keep in your own notes):

1. Does time travel copy all data files each time, or mainly change which files the log points to?
2. Why does RESTORE add a new version instead of deleting old history?
3. Give one risk of running VACUUM with very short retention.
4. Does OPTIMIZE change logical query results, or mainly file layout?
5. When is ZORDER most useful?
6. Which table property enables Change Data Feed?
7. What happens to a write that violates a CHECK constraint?

---

## Section 12 — Liquid clustering / predictive optimization (brief note)

Newer runtimes may offer liquid clustering or automated maintenance. Check your Databricks runtime release notes for what applies in your workspace.

---

## End of notebook 03

Continue with `04-Day5-Delta-DML-MERGE-SCD.ipynb` for MERGE, UPDATE, DELETE, and SCD patterns.